In [ ]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-63c29bd1-06a4-380f-f941-2becdceb9612)


In [ ]:
!conda --version


/bin/bash: conda: command not found


In [ ]:
# run this cell if you want to install conda on colab
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/latest/download/Mambaforge-colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:26
🔁 Restarting kernel...


In [ ]:
!conda install -c pytorch faiss-gpu

Solving environment: / - \ | / - \ | / - \ | / failed with initial frozen solve. Retrying with flexible solve.
Solving environment: - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - faiss-gpu


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2022.12.7  |       ha878542_0         143 KB  conda-forge
    certifi

In [ ]:
!conda install -c conda-forge sentence-transformers

Solving environment: / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / 

In [ ]:
# !conda install pytorch torchvision torchaudio pytorch-cuda=11.6 -c pytorch -c nvidia

Solving environment: - failed with initial frozen solve. Retrying with flexible solve.
Solving environment: - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | 

In [ ]:
from sentence_transformers import SentenceTransformer, util
import os
import csv
import pickle
import time
import faiss
import numpy as np

In [ ]:
import json
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import time
import gzip
import os
import torch

if not torch.cuda.is_available():
  print("Warning: No GPU found. Please add GPU to your notebook")


# We use the Bi-Encoder to encode all passages, so that we can use it with sematic search
model_name = 'nq-distilbert-base-v1'
bi_encoder = SentenceTransformer(model_name)
top_k = 5  # Number of passages we want to retrieve with the bi-encoder

Downloading:   0%|          | 0.00/690 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/190 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/540 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

Downloading:   0%|          | 0.00/112 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/466k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/554 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/232k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/229 [00:00<?, ?B/s]

In [ ]:
# As dataset, we use Simple English Wikipedia. Compared to the full English wikipedia, it has only
# about 170k articles. We split these articles into paragraphs and encode them with the bi-encoder

wikipedia_filepath = 'data/simplewiki-2020-11-01.jsonl.gz'

if not os.path.exists(wikipedia_filepath):
    util.http_get('http://sbert.net/datasets/simplewiki-2020-11-01.jsonl.gz', wikipedia_filepath)

passages = []
with gzip.open(wikipedia_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        data = json.loads(line.strip())
        for paragraph in data['paragraphs']:
            # We encode the passages as [title, text]
            passages.append([data['title'], paragraph])

# If you like, you can also limit the number of passages you want to use
print("Passages:", len(passages))

In [ ]:
dataset_path = '/content/train_data.csv'
embedding_cache_path = 'embeddings-{}-size-.pkl'.format(model_name.replace('/', '_'))


In [ ]:
passages = []

In [ ]:
with open(dataset_path,'rt', encoding='utf8', ) as fIn:
    fields = ['ID', 'Theme', 'Paragraph', 'Question','Answer_possible', 'Answer_text', 'Answer_start']
    reader = csv.DictReader(fIn,skipinitialspace=True,fieldnames=fields, delimiter = ",", quotechar='"')
    next(reader) # skip the heading
    for row in reader:
        passages.append([row['Theme'],row['Paragraph']])
        # print(row['Theme'],row['Paragraph'])
        

In [ ]:
if not os.path.exists(dataset_path):
    print("Download dataset")

# Get all unique sentences from the file
corpus_sentences = set()
with open(dataset_path,'rt', encoding='utf8', ) as fIn:
    fields = ['ID', 'Theme', 'Paragraph', 'Question','Answer_possible', 'Answer_text', 'Answer_start']
    reader = csv.DictReader(fIn,skipinitialspace=True,fieldnames=fields, delimiter = ",", quotechar='"')
    next(reader) # skip the heading
    for row in reader:
        sentence = row['Theme'] + "," + row['Paragraph']
        corpus_sentences.add(sentence)

corpus_sentences = list(corpus_sentences)


In [ ]:
passages

In [ ]:
corpus_sentences

In [ ]:
print("Encode the corpus. This might take a while")
corpus_embeddings = bi_encoder.encode(corpus_sentences, show_progress_bar=True, convert_to_numpy=True)

print("Store file on disc")
with open(embedding_cache_path, "wb") as fOut:
    pickle.dump({'sentences': corpus_sentences, 'embeddings': corpus_embeddings}, fOut)

Encode the corpus. This might take a while


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Store file on disc


In [ ]:
passages[1]

['Aileen Wuornos',
 'Aileen Carol Wuornos Pralle (born Aileen Carol Pittman; February 29, 1956\xa0– October 9, 2002) was an American serial killer. She was born in Rochester, Michigan. She confessed to killing six men in Florida and was executed in Florida State Prison by lethal injection for the murders. Wuornos said that the men she killed had raped her or tried to rape her while she was working as a prostitute.']

In [ ]:
# To speed things up, pre-computed embeddings are downloaded.
# The provided file encoded the passages with the model 'nq-distilbert-base-v1'
if model_name == 'nq-distilbert-base-v1':
    embeddings_filepath = 'simplewiki-2020-11-01-nq-distilbert-base-v1.pt'
    if not os.path.exists(embeddings_filepath):
        util.http_get('http://sbert.net/datasets/simplewiki-2020-11-01-nq-distilbert-base-v1.pt', embeddings_filepath)

    corpus_embeddings = torch.load(embeddings_filepath)
    corpus_embeddings = corpus_embeddings.float()  # Convert embedding file to float
    if torch.cuda.is_available():
        corpus_embeddings = corpus_embeddings.to('cuda')
else:  # Here, we compute the corpus_embeddings from scratch (which can take a while depending on the GPU)
    corpus_embeddings = bi_encoder.encode(passages, convert_to_numpy=True, show_progress_bar=True)

  0%|          | 0.00/783M [00:00<?, ?B/s]

In [ ]:
corpus_embeddings.shape

torch.Size([509663, 768])

In [ ]:
#Defining our FAISS index
#Number of clusters used for faiss. Select a value 4*sqrt(N) to 16*sqrt(N) - https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index
embedding_size = 768    #Size of embeddings
n_clusters = 1024

#We use Inner Product (dot-product) as Index. We will normalize our vectors to unit length, then is Inner Product equal to cosine similarity
quantizer = faiss.IndexFlatIP(embedding_size)
index = faiss.IndexIVFFlat(quantizer, embedding_size, n_clusters, faiss.METRIC_INNER_PRODUCT)

#Number of clusters to explorer at search time. We will search for nearest neighbors in 3 clusters.
index.nprobe = 3

In [ ]:
# # rough code


# model_name = 'quora-distilbert-multilingual'
# model = SentenceTransformer(model_name)

# url = "http://qim.fs.quoracdn.net/quora_duplicate_questions.tsv"
# dataset_path = "quora_duplicate_questions.tsv"
# max_corpus_size = 100000

# embedding_cache_path = 'quora-embeddings-{}-size-{}.pkl'.format(model_name.replace('/', '_'), max_corpus_size)


# embedding_size = 768    #Size of embeddings
# top_k_hits = 10         #Output k hits

# #Defining our FAISS index
# #Number of clusters used for faiss. Select a value 4*sqrt(N) to 16*sqrt(N) - https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index
# n_clusters = 1024

# #We use Inner Product (dot-product) as Index. We will normalize our vectors to unit length, then is Inner Product equal to cosine similarity
# quantizer = faiss.IndexFlatIP(embedding_size)
# index = faiss.IndexIVFFlat(quantizer, embedding_size, n_clusters, faiss.METRIC_INNER_PRODUCT)

# #Number of clusters to explorer at search time. We will search for nearest neighbors in 3 clusters.
# index.nprobe = 3

# #Check if embedding cache path exists
# if not os.path.exists(embedding_cache_path):
#     # Check if the dataset exists. If not, download and extract
#     # Download dataset if needed
#     if not os.path.exists(dataset_path):
#         print("Download dataset")
#         util.http_get(url, dataset_path)

#     # Get all unique sentences from the file
#     corpus_sentences = set()
#     with open(dataset_path, encoding='utf8') as fIn:
#         reader = csv.DictReader(fIn, delimiter='\t', quoting=csv.QUOTE_MINIMAL)
#         for row in reader:
#             corpus_sentences.add(row['question1'])
#             if len(corpus_sentences) >= max_corpus_size:
#                 break

#             corpus_sentences.add(row['question2'])
#             if len(corpus_sentences) >= max_corpus_size:
#                 break

#     corpus_sentences = list(corpus_sentences)
#     print("Encode the corpus. This might take a while")
#     corpus_embeddings = model.encode(corpus_sentences, show_progress_bar=True, convert_to_numpy=True)

#     print("Store file on disc")
#     with open(embedding_cache_path, "wb") as fOut:
#         pickle.dump({'sentences': corpus_sentences, 'embeddings': corpus_embeddings}, fOut)
# else:
#     print("Load pre-computed embeddings from disc")
#     with open(embedding_cache_path, "rb") as fIn:
#         cache_data = pickle.load(fIn)
#         corpus_sentences = cache_data['sentences']
#         corpus_embeddings = cache_data['embeddings']



KeyboardInterrupt: ignored

In [ ]:
corpus_embeddings.shape


torch.Size([509663, 768])

In [ ]:
len(corpus_sentences)

9416

In [ ]:
type(corpus_embeddings)

torch.Tensor

In [ ]:
### Create the FAISS index
print("Start creating FAISS index")
# First, we need to normalize vectors to unit length
corpus_embeddings = corpus_embeddings.cpu() / np.linalg.norm(corpus_embeddings.cpu(), axis=1)[:, None]

# Then we train the index to find a suitable clustering
index.train(corpus_embeddings.cpu().detach().numpy())

# Finally we add all embeddings to the index
index.add(corpus_embeddings.cpu().detach().numpy())





Start creating FAISS index


AttributeError: ignored

In [ ]:
top_k

5

In [ ]:
top_k_hits = 10

In [ ]:
######### Search in the index ###########


print("Corpus loaded with {} sentences / embeddings".format(len(passages)))

while True:
    inp_question = input("Please enter a question: ")

    start_time = time.time()
    question_embedding = bi_encoder.encode(inp_question)

    #FAISS works with inner product (dot product). When we normalize vectors to unit length, inner product is equal to cosine similarity
    question_embedding = question_embedding / np.linalg.norm(question_embedding)
    question_embedding = np.expand_dims(question_embedding, axis=0)

    # Search in FAISS. It returns a matrix with distances and corpus ids.
    distances, corpus_ids = index.search(question_embedding, top_k_hits)

    # We extract corpus ids and scores for the first query
    hits = [{'corpus_id': id, 'score': score} for id, score in zip(corpus_ids[0], distances[0])]
    hits = sorted(hits, key=lambda x: x['score'], reverse=True)
    end_time = time.time()

    print("Input question:", inp_question)
    print("Results (after {:.3f} seconds):".format(end_time-start_time))
    for hit in hits[0:top_k_hits]:
        print("\t{:.3f}\t{}".format(hit['score'], corpus_sentences[hit['corpus_id']]))

    # Approximate Nearest Neighbor (ANN) is not exact, it might miss entries with high cosine similarity
    # Here, we compute the recall of ANN compared to the exact results
    correct_hits = util.semantic_search(question_embedding, corpus_embeddings, top_k=top_k_hits)[0]
    correct_hits_ids = set([hit['corpus_id'] for hit in correct_hits])

    ann_corpus_ids = set([hit['corpus_id'] for hit in hits])
    if len(ann_corpus_ids) != len(correct_hits_ids):
        print("Approximate Nearest Neighbor returned a different number of results than expected")

    recall = len(ann_corpus_ids.intersection(correct_hits_ids)) / len(correct_hits_ids)
    print("\nApproximate Nearest Neighbor Recall@{}: {:.2f}".format(top_k_hits, recall * 100))

    if recall < 1:
        print("Missing results:")
        for hit in correct_hits[0:top_k_hits]:
            if hit['corpus_id'] not in ann_corpus_ids:
                print("\t{:.3f}\t{}".format(hit['score'], passages[hit['corpus_id']]))
    print("\n\n========\n")

Corpus loaded with 84850 sentences / embeddings
Please enter a question: When did Beyonce leave Destiny's Child and become a solo singer?
Input question: When did Beyonce leave Destiny's Child and become a solo singer?
Results (after 0.013 seconds):


IndexError: ignored